# Deliverable 4 — Demo de Video: Base vs `Redactor` QLoRA con Mensajes Intermedios

Cuaderno auxiliar para grabar la demostración (≤ 3 min) requerida por `Deliverable 4`:
1. Visualiza los **mensajes intermedios entre agentes** (`Enrutador` → `Recuperador` → `Redactor` → `Verificador`).
2. Compara **lado a lado** el sistema completamente Base (`LoRA OFF` en los 4 agentes) frente al sistema D4 con `Pablox0333/qwen3-4b-redactor-udec-d4` activo **solo en `Redactor`**.
3. Permite modo **instantáneo desde las trazas oficiales** (`evaluacion/comparativa_base_vs_finetuned.json`) o **ejecución en vivo en GPU** sobre Colab T4/L4 (`--live`).

## 1. Clonar repositorio (funciona en CPU o GPU para replay; GPU solo para modo `--live`)

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
RAMA_GIT = "deliverable-4"
WORKDIR = Path("/content/d4_video_demo")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", RAMA_GIT, REPO_URL, str(WORKDIR)],
    check=True,
)

VIDEO_DIR = WORKDIR / "Deliverable 4" / "video"
if str(VIDEO_DIR) not in sys.path:
    sys.path.insert(0, str(VIDEO_DIR))

print("Commit activo:", subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "--short", "HEAD"], text=True).strip())

## 2. Tarjetas comparativas Base vs Fine-tuned con intercambios intermedios (`H07` y `H14`)
Muestra las ejecuciones reales emparejadas en T4 con cada paso del `Enrutador`, `Recuperador`, `Redactor` y `Verificador` desplegable.

In [ ]:
from IPython.display import HTML, display
import mostrar_demo_d4 as demo

comparativas = demo.cargar_comparativas(WORKDIR / "Deliverable 4" / "evaluacion" / "comparativa_base_vs_finetuned.json")
for qid in ("dificil_25:H07", "dificil_25:H14"):
    display(HTML(demo.renderizar_comparativa_html(comparativas[qid])))

## 3. Transparencia de fallos: degradación por Verificador base (`H19`) y persistencia (`H03`)

In [ ]:
fallos = demo.cargar_fallos(WORKDIR / "Deliverable 4" / "evaluacion" / "casos_fallo_y_regresion.json")
print(demo.formatear_fallo_texto(fallos["dificil_25:H19"], mostrar_traza=True))

## 4. (Opcional en vivo con GPU T4/L4) Re-ejecutar Base vs FT en vivo sobre `dificil_25:H07`
Demora ~4–5 min en T4 (`~228 s` Base en 3 ciclos + `~79 s` FT en 1 ciclo). Úsalo si quieres mostrar inferencia fresca durante la grabación.

In [ ]:
# Descomentar cuando el entorno tenga GPU e instalar dependencias primero:
# !pip install -q -U "transformers>=4.51" peft bitsandbytes accelerate "huggingface-hub>=0.30,<2.0" sentence-transformers
# !python3 "{WORKDIR / 'Deliverable 4' / 'video' / 'mostrar_demo_d4.py'}" --live --qid dificil_25:H07 --output-dir /content/live_demo_h07